In [4]:
from __future__ import annotations

from langchain_groq import ChatGroq
import operator
import os
import re
from datetime import date, timedelta
from pathlib import Path
from typing import TypedDict, List, Optional, Literal, Annotated

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv

In [5]:
class Task(BaseModel):
    id : int
    title : str
    goal : str = Field(... , description="One sentence describing what the reader should do/understand")
    # Field(...)  → "Required field"
    # Field(x)    → "Optional (Default value = x)"
    # Field()     → "Optional (Default = None)"
    # name: str , this is required by default , no need for ...
    bullets : List[str] = Field(... , min_length=3 , max_length=6)
    target_words : int = Field(... , description="Target words (120 - 550).")

    tags : List[str] = Field(default_factory=list)
    # if u do tags : List[str] = []
    # every instance of this class will get the same list , every instance shares the same list , so if u change the list via one instance , it will affect others.
    # for each instance to have their separate list we use default_factory
    # Each workflow run gets its own memory list
    requires_research : bool = False
    requires_citations : bool = False
    requires_code : bool = False


class Plan(BaseModel):
    blog_title : str
    audience : str
    tone : str
    blog_kind: Literal["explainer", "tutorial", "news_roundup", "comparison", "system_design"] = "explainer"
    constraints: List[str] = Field(default_factory=list)
    tasks : List[Task]


class EvidenceItem(BaseModel):
    title : str
    url : str
    published_at : Optional[str] = None
    snippet : Optional[str] = None
    source : Optional[str] = None


class RouterDecision(BaseModel):
    needs_research : bool
    mode: Literal["closed_book", "hybrid", "open_book"]
    reason : str
    queries : List[str] = Field(default_factory=list)
    max_results_per_query: int = Field(5)


class EvidencePack(BaseModel):
    evidence: List[EvidenceItem] = Field(default_factory=list)


class ImageSpec(BaseModel):
    placeholder : str = Field(... , description="e.g. [[IMAGE_1]]")
    filename : str = Field(..., description="Save under images/, e.g. picture.png")
    alt : str
    caption : str
    prompt : str = Field(... , description = "Prompt to send to the image model")
    size: Literal["1024x1024", "1024x1536", "1536x1024"] = "1024x1024"
    quality: Literal["low", "medium", "high"] = "medium"


class GlobalImagePlan(BaseModel):
    md_with_placeholders: str
    images: List[ImageSpec] = Field(default_factory=list)





class State(TypedDict):
    topic: str


    mode: str
    needs_research: bool
    queries: List[str]
    evidence: List[EvidenceItem]
    plan: Optional[Plan]


    as_of: str
    recency_days: int

    # workers
    sections: Annotated[List[tuple[int, str]], operator.add]  # (task_id, section_md)
    # Annotated[type , metadata]

    # if 2 nodes sets 
    # {"sections": [(1, "Intro")]}
    # {"sections": [(2, "Body")]}

    # sections = [(2, "Body")]   # ❌ overwrites previous

    # With operator.add
    # Now:
    
    # sections = [(1, "Intro")] + [(2, "Body")]
    # 👉 Result:
    # [(1, "Intro"), (2, "Body")]


    
    merged_md: str
    md_with_placeholders: str
    image_specs: List[dict]

    final: str




    

In [6]:
llm = ChatGroq(
        api_key="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",
        model="openai/gpt-oss-120b",
        )

In [7]:
ROUTER_SYSTEM = """You are a routing module for a technical blog planner.

Decide whether web research is needed BEFORE planning.

Modes:
- closed_book (needs_research=false): evergreen concepts.
- hybrid (needs_research=true): evergreen + needs up-to-date examples/tools/models.
- open_book (needs_research=true): volatile weekly/news/"latest"/pricing/policy.

If needs_research=true:
- Output 3–10 high-signal, scoped queries.
- For open_book weekly roundup, include queries reflecting last 7 days.
"""

In [ ]:
def router_node(state : State) -> State:
    structured_llm = llm.with_structured_output(RouterDecision)

    decision : RouterDecision = structured_llm.invoke(
        [
            SystemMessage(content = ROUTER_SYSTEM),
            HumanMessage(content = f"Topic : {state['topic']}\n As-of date: {state['as_of']}")            
        ]
    )

    if decision.node == "open_book":
        recency_days = 7
    elif decision.node == "hybrid":
        recency_days = 45
    else:
        recency_days = 3650

    return{
        "needs_research" : decision.needs_research,
        "mode" : decision.mode,
        "queries" : decision.queries,
        "recency_days" : recency_days
    }


def route_next(state : State) -> str:
    return "research" if state["needs_research"] else "orchestrator"


def tavily_search(query : str , max_results : int = 5) -> List[dict]:
    try:
        from langchain.community.tools.tavily_search import TavilySearchResults
        tool = TavilySearchResults(max_results = max_results)
        results = tool.invoke({"query" : query})

        out : List[dict] = []

        for r in results:
            out.append(
                {
                    "title": r.get("title") or "",
                    "url": r.get("url") or "",
                    "snippet": r.get("content") or r.get("snippet") or "",
                    "published_at": r.get("published_date") or r.get("published_at"),
                    "source": r.get("source"),
                }
            )
        return out
    except Exception:
        return []
    

def _iso_to_date(s: Optional[str]) -> Optional[date]:
    if not s:
        return None
    try:
        return date.fromisoformat(s[:10])
    except Exception:
        return None



In [9]:

RESEARCH_SYSTEM = """You are a research synthesizer.

Given raw web search results, produce EvidenceItem objects.

Rules:
- Only include items with a non-empty url.
- Prefer relevant + authoritative sources.
- Normalize published_at to ISO YYYY-MM-DD if reliably inferable; else null (do NOT guess).
- Keep snippets short.
- Deduplicate by URL.
"""

In [ ]:
def research_node(state : State) -> State:
    queries = state['queries'] or []
    